In [1]:
import os, io, time, json, hashlib, pathlib, sys
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import re
import matplotlib.pyplot as plt
from urllib.parse import urlparse
from datetime import datetime, timedelta
from geoquant.configs.config import *
# from data_io import *
from geoquant.series_utils import *
from geoquant.books import IBKR_live, extract_target_weights



STOOQ_API: rjpadqeTh6Uc8xu7F0D52AKXVsNEL9rC


In [2]:
z = extract_target_weights(IBKR_live)
print(z)

{'XMWX': 0.3, 'EMIM': 0.25, 'GWX': 0.15, 'VUAG': 0.1, 'SGLN': 0.08, 'BATG': 0.12, 'YCA': 0.5, 'CASH_CHF': 0.0, 'CASH_JPY': 0.0}


In [2]:
fx_map = make_fx_map(url, holdings, params, max_age, no_fx=False, usd_shift=True)
#  DOES IT GIVE ALL THE FX RATES 
need = {h['ccy'] for h in holdings if h['ccy'].upper() != 'CHF'}
got = set(fx_map)
assert all(c+'CHF' in got for c in need), f"Missing FX rates: {need - {c[:-3] for c in got if c.endswith('CHF')}}"

# print(fx_map)

for k in fx_map:
    s = fx_map[k]
    # allow one trailing NaN due to T+1 shifting or end-of-window
    if s.isna().sum() == 1 and pd.isna(s.iloc[-1]):
        s = s.iloc[:-1]
    assert isinstance(s, pd.Series)
    print(len(s))
    assert s.index.is_monotonic_increasing and len(s) > 5
    # print("NaNs:", int(s.isna().sum()))
    # print(s[s.isna()].head(5))
    # print("first/last valid:", s.first_valid_index(), s.last_valid_index())
    assert s.notna().all(), f"FX {s} has NaNs; upstream fetch/parse problem"

-------------fetching currencies-------------
    Applied USDCHF T+1 shift
1512
1507


In [3]:
ccy='CHF'
s = deal_with_cash(ccy, fx_map, lookback_days=50)

assert isinstance(s, pd.Series)
assert s.index.is_monotonic_increasing and len(s) > 5
assert s.notna().all(), f"cash |{ccy} has Nans"
if ccy == 'CHF':
    # check all values are 1
    assert (s==1).all(), "CHF not all 1's"

In [4]:
print(f'holdings[0]={holdings[0]}')

holdings[0]={'name': 'EMIM', 'ticker': 'EMIM.LSE', 'ccy': 'GBP', 'gbx': True, 'position': 100}


In [5]:
asset_s=fetch_asset_series(url, holdings[0],fx_map, params, max_age, lookback_days) 

print(asset_s)
assert isinstance(asset_s, pd.Series)
assert asset_s.index.is_monotonic_increasing and len(s) > 5


Date
2020-01-02    2331.25
2020-01-03    2322.50
2020-01-06    2291.00
2020-01-07    2301.50
2020-01-08    2305.75
               ...   
2025-09-15    3151.00
2025-09-16    3154.00
2025-09-17    3179.00
2025-09-18    3190.00
2025-09-19    3200.00
Name: Adjusted_close, Length: 1444, dtype: float64


In [7]:
def TESTRETURN_fetch_asset_series(asset_s, *, gbx=False, ticker=None):
    # Positive prices
    assert (asset_s > 0).all(), "Non-positive prices found"

    # Sanity on gaps vs business days
    # Max calendar-day gap between observations (holidays/weekends are fine; large gaps are suspicious)
    idx = asset_s.index
    gaps = idx.to_series().diff().dt.days.iloc[1:]
    assert int(gaps.max()) <= 5, f"Unusually long gap: {int(gaps.max())} days"

    # Outlier return guard (catches unadjusted splits/bad merges)
    rets =asset_s.pct_change().dropna()
    extreme = (rets.abs() > 0.3).sum()  # 30% daily move thresholasset_s
    assert extreme <= 2, f"Unusual number of large moves (>30%): {extreme}"

    # gbx scaling sanity (if flagged as pence -> expect post-scale prices not in the thousands)
    if gbx:
        med = float(s.median())
        assert med < 1000, f"Looks like pence (median {med}); did you divide by 100?"

for h in holdings:
    # print(f'Checking holding {h}')  

    asset_s=fetch_asset_series(url, h,fx_map, params, max_age, lookback_days) 
    if h['name'].startswith('CASH_'):
        # print(f"cash {h['name']}"  )      
        asset_s = asset_s.iloc[:-1]  # trim to lookback window
        if asset_s.all() != 1.0:
            print(f"cash {h['name']} not all 1's")
        assert (asset_s > 0).all(), f"Non-positive cash {h['name']} found"
    TESTRETURN_fetch_asset_series(asset_s, gbx=h.get("gbx", False), ticker=h.get("ticker"))

# v